In [3]:
# Import standard dependencies
import cv2
import os
import random
import uuid
import glob
import math
import numpy as np
from matplotlib import pyplot as plt

In [4]:
# Import PyTorch dependencies
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [9]:
def generate_triplets(i_dir='i', v_dir='v', num_triplets_per_anchor=5):
    """Generate triplets with multiple pos/neg pairs per anchor for better diversity."""
    triplets = []
    if os.path.exists(i_dir) and os.path.exists(v_dir):
        for i_folder in os.listdir(i_dir):
            base_id = i_folder
            anchor_imgs = glob.glob(os.path.join(i_dir, i_folder, '*.*'))
            
            pos_folders = [f for f in os.listdir(v_dir) if f.startswith(base_id + '-')]
            pos_imgs = []
            for f in pos_folders:
                pos_imgs.extend(glob.glob(os.path.join(v_dir, f, '*.*')))
                
            neg_folders = [f for f in os.listdir(v_dir) if not f.startswith(base_id + '-')]
            neg_imgs = []
            for f in neg_folders:
                neg_imgs.extend(glob.glob(os.path.join(v_dir, f, '*.*')))
                
            for anc in anchor_imgs:
                if pos_imgs and neg_imgs:
                    for _ in range(min(num_triplets_per_anchor, len(pos_imgs))):
                        pos = random.choice(pos_imgs)
                        neg = random.choice(neg_imgs)
                        triplets.append((anc, pos, neg))
    else:
        print('Directories i and v not found.')
    return triplets

triplets = generate_triplets()
print(f'Total triplets generated: {len(triplets)}')


Total triplets generated: 150000


In [10]:
# Data augmentation
data_augmentation = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(
        size=(112, 112),
        scale=(0.9, 1.0)
    ),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor()
])
test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((112, 112)), # Just resize, no random cropping
    transforms.ToTensor()
])

In [11]:
def preprocess(file_path, transform=None):
    byte_img = cv2.imread(file_path)
    img = cv2.cvtColor(byte_img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (112, 112))
    if transform:
        img = transform(img)
    return img


In [12]:
data = triplets


In [13]:
#Creating Data Pipeline
class TripletDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        anc_path, pos_path, neg_path = self.data[index]

        return (
            preprocess(anc_path, self.transform),
            preprocess(pos_path, self.transform),
            preprocess(neg_path, self.transform)
        )


In [14]:
random.shuffle(data)


In [15]:
train_size = round(len(data) * .7)
test_size = len(data) - train_size
train_data_raw, test_data_raw = random_split(data, [train_size, test_size])

train_dataset = TripletDataset(train_data_raw, transform=data_augmentation)
test_dataset = TripletDataset(test_data_raw, transform=test_transform)

train_data = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
test_data = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=0)


In [16]:
class Embedding(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=10)
        self.bn1 = nn.BatchNorm2d(64)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=7)
        self.bn2 = nn.BatchNorm2d(128)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(128, 128, kernel_size=4)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2, 2)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=4)
        self.bn4 = nn.BatchNorm2d(256)
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(256 * 6 * 6, 512)

    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))
        x = F.relu(self.bn4(self.conv4(x)))
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.fc1(x)
        x = F.normalize(x, p=2, dim=1)
        return x


mod = Embedding()

In [17]:
class SiameseNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = Embedding()

    def forward(self, x1, x2=None, x3=None):
        if x2 is None and x3 is None:
            return self.embedding(x1)
        return self.embedding(x1), self.embedding(x2), self.embedding(x3)

siamese_network = SiameseNetwork().to(device)
print(siamese_network)


SiameseNetwork(
  (embedding): Embedding(
    (conv1): Conv2d(3, 64, kernel_size=(10, 10), stride=(1, 1))
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (conv2): Conv2d(64, 128, kernel_size=(7, 7), stride=(1, 1))
    (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (conv3): Conv2d(128, 128, kernel_size=(4, 4), stride=(1, 1))
    (bn3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (conv4): Conv2d(128, 256, kernel_size=(4, 4), stride=(1, 1))
    (bn4): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (flatten): Flatten(st

In [18]:
triplet_loss = nn.TripletMarginLoss(margin=1.0, p=2)
opt = torch.optim.Adam(siamese_network.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50, eta_min=1e-6)


In [19]:
checkpoint_dir = './training_checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_prefix = os.path.join(checkpoint_dir, 'ckpt')

def save_checkpoint(path):
    torch.save({
        'model_state_dict': siamese_network.state_dict(),
        'optimizer_state_dict': opt.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
    }, path)

def load_checkpoint(path):
    ckpt = torch.load(path, map_location=device)
    siamese_network.load_state_dict(ckpt['model_state_dict'])
    opt.load_state_dict(ckpt['optimizer_state_dict'])
    if 'scheduler_state_dict' in ckpt:
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])

In [59]:
def train_step(batch):
    anc, pos, neg = batch[0].to(device), batch[1].to(device), batch[2].to(device)

    siamese_network.train()
    opt.zero_grad()

    emb_a, emb_p, emb_n = siamese_network(anc, pos, neg)
    loss = triplet_loss(emb_a, emb_p, emb_n)
    
    loss.backward()
    opt.step()

    # Compute accuracy metric for batch (dist(A, P) < dist(A, N))
    dist_pos = F.pairwise_distance(emb_a, emb_p, 2)
    dist_neg = F.pairwise_distance(emb_a, emb_n, 2)
    correct = (dist_pos < dist_neg).sum().item()
    
    return loss, correct


In [60]:
from tqdm import tqdm

def train(train_dataloader, val_dataloader, EPOCHS):
    for epoch in range(1, EPOCHS + 1):
        print(f"\nEpoch {epoch}/{EPOCHS} (LR: {scheduler.get_last_lr()[0]:.6f})")

        total_correct = 0
        total_samples = 0

        for batch in tqdm(train_dataloader, desc=f"Epoch {epoch}/{EPOCHS} (Train)"):
            loss, correct = train_step(batch)
            total_correct += correct
            total_samples += batch[0].size(0)

        train_acc = total_correct / total_samples if total_samples > 0 else 0
        print(f"Train Loss: {loss.item():.4f}, Train Acc: {train_acc:.4f}")

        val_correct = 0
        val_samples = 0
        siamese_network.eval()
        with torch.no_grad():
            for val_batch in tqdm(val_dataloader, desc=f"Epoch {epoch}/{EPOCHS} (Val)"):
                anc, pos, neg = val_batch[0].to(device), val_batch[1].to(device), val_batch[2].to(device)
                emb_a, emb_p, emb_n = siamese_network(anc, pos, neg)
                dist_pos = F.pairwise_distance(emb_a, emb_p, 2)
                dist_neg = F.pairwise_distance(emb_a, emb_n, 2)
                val_correct += (dist_pos < dist_neg).sum().item()
                val_samples += anc.size(0)
        
        val_acc = val_correct / val_samples if val_samples > 0 else 0
        print(f"Validation Acc: {val_acc:.4f}")

        # Step the learning rate scheduler
        scheduler.step()

        if epoch % 10 == 0:
            save_checkpoint(checkpoint_prefix + f"-{epoch}.pt")


In [61]:
train(train_data, test_data, EPOCHS=20)



Epoch 1/20 (LR: 0.000100)


Epoch 1/20 (Train): 100%|██████████| 6563/6563 [19:31<00:00,  5.60it/s]


Train Loss: 0.5839, Train Acc: 0.7914


Epoch 1/20 (Val): 100%|██████████| 2813/2813 [03:08<00:00, 14.93it/s]


Validation Acc: 0.8426

Epoch 2/20 (LR: 0.000100)


...

Validation Acc: 0.9242

Epoch 20/20 (LR: 0.000069)


Epoch 20/20 (Train): 100%|██████████| 6563/6563 [21:23<00:00,  5.11it/s]


Train Loss: 0.1326, Train Acc: 0.9238


Epoch 20/20 (Val): 100%|██████████| 2813/2813 [03:12<00:00, 14.63it/s]


Validation Acc: 0.9234


In [62]:
torch.save(siamese_network.state_dict(), "siamese_modelv-3.pth")

In [20]:
model=SiameseNetwork().to(device)
model.load_state_dict(torch.load("siamese_modelv-3.pth",map_location=device))

<All keys matched successfully>

In [21]:
ANC_PATH = os.path.join('testing', 'input')
POS_PATH = os.path.join('testing', 'validation')

In [22]:
# NOTE: Must match training size (112x112), NOT 100x100!
def preprocess(file_path, transform=None):
    byte_img = cv2.imread(file_path)
    img = cv2.cvtColor(byte_img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (112, 112))
    if transform:
        img = transform(img)
    return img


In [23]:
def verify(model, input_image_path, POS_PATH, n=5, verification_threshold=0.6):
    model.eval()
    
    input_img = preprocess(input_image_path, transform=test_transform).unsqueeze(0).to(device)
    
    folder_averages = {}
    
    # Iterate through folders inside validation path
    for folder_name in os.listdir(POS_PATH):
        folder_path = os.path.join(POS_PATH, folder_name)
        if not os.path.isdir(folder_path):
            continue
            
        results = []
        for val_filename in os.listdir(folder_path):
            val_img_path = os.path.join(folder_path, val_filename)
            if not val_filename.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                continue
            
            val_img = preprocess(val_img_path, transform=test_transform).unsqueeze(0).to(device)
            with torch.no_grad():
                # Get embeddings for input and validation images
                emb_input = model(input_img)
                emb_val = model(val_img)
                
                # Calculate Euclidean distance
                dist = F.pairwise_distance(emb_input, emb_val, 2).item()
                
                # Convert distance to similarity score (0 to 1)
                similarity = 1.0 / (1.0 + dist)
                results.append(similarity)
        
        if len(results) > 0:
            results.sort(reverse=True)
            top_n = results[:50] # Matches original notebook logic
            folder_averages[folder_name] = sum(top_n) / len(top_n)
            
    best_match = None
    highest_avg = 0
    
    if folder_averages:
        best_match = max(folder_averages, key=folder_averages.get)
        highest_avg = folder_averages[best_match]
        
    if highest_avg >= verification_threshold:
        verified_identity = best_match
    else:
        verified_identity = "Unverified"
        
    return verified_identity, highest_avg, folder_averages


In [25]:
all_predictions = []
for filename in os.listdir(ANC_PATH):
    if filename.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
        input_image_path = os.path.join(ANC_PATH, filename)
        identity, score, details = verify(model, input_image_path, POS_PATH, verification_threshold=0.6)
        print(f"Image: {filename} -> Identity: {identity} (Score: {score:.4f})")
        all_predictions.append((filename, identity, score))
all_predictions


Image: 0f992e75-994f-11f1-802e-14ac603236d0.jpg -> Identity: Lokesh (Score: 0.6589)


[('0f992e75-994f-11f1-802e-14ac603236d0.jpg', 'Lokesh', 0.6588581537210704)]

In [68]:
if all_predictions:
    avg_score = np.mean([p[2] for p in all_predictions])
    print(f"Average prediction score: {avg_score:.4f}")


Average prediction score: 0.6180
